# NB2: GM Pipeline (First LLM Call)

**Purpose:** Test whether the GM can produce calibrated probabilities given domain models and base rates.

**Tests:**
1. Five diverse ActionIntents adjudicated through the GM pipeline
2. All AdjudicationPackets parse successfully (Pydantic strict mode)
3. Probabilities sum to 1.0
4. All var_ids exist in the scenario schema
5. GM reasoning references domain models (not generic vibes)
6. GM probabilities within ±0.15 of mechanical base rate

**Resolves:** U1 (probability calibration), U2 (context size), U8 (base rate anchoring)

In [ ]:
import sys
sys.path.insert(0, '..')

import json
from wargame.scenario import load_scenario, init_db
from wargame.engine import get_all_variables
from wargame.models import ActionIntent, AdjudicationPacket
from wargame.gm import (
    select_relevant_domain_models,
    compute_mechanical_base_rate,
    build_gm_messages,
    validate_adjudication,
    normalize_probabilities,
)

SCENARIO_PATH = '../scenarios/us_iran_2026.yaml'
spec = load_scenario(SCENARIO_PATH)
conn = init_db(spec)
state = get_all_variables(conn)

# Derived constants
valid_var_ids = {sv.id for sv in spec.state_variables}
valid_actor_ids = {a.id for a in spec.actors}
actor_ids = list(valid_actor_ids)
variable_ids = list(valid_var_ids)

print(f'Loaded scenario: {spec.meta.name}')
print(f'Variables: {len(valid_var_ids)}, Actors: {len(valid_actor_ids)}')

## Define 5 Diverse Test Actions

One of each major category to test breadth.

In [ ]:
test_actions = [
    ActionIntent(
        actor_id='actor_us',
        action_category='covert',
        target_entities=['actor_iran'],
        instruments_used=['inst_us_cyber'],
        intended_effect='Deploy cyberweapon to degrade Iranian centrifuge performance at Natanz, delaying nuclear breakout.',
        resource_cost=3,
        ambiguity_flags=['deniable_attribution'],
    ),
    ActionIntent(
        actor_id='actor_us',
        action_category='economic',
        target_entities=['actor_iran'],
        instruments_used=['inst_us_sanctions'],
        intended_effect='Impose new secondary sanctions on Iranian oil exports to further squeeze regime revenue.',
        resource_cost=2,
    ),
    ActionIntent(
        actor_id='actor_iran',
        action_category='diplomatic',
        target_entities=['actor_us'],
        instruments_used=['inst_iran_diplomacy'],
        intended_effect='Open backchannel diplomatic contact through Oman to propose limited enrichment freeze in exchange for sanctions relief.',
        resource_cost=2,
    ),
    ActionIntent(
        actor_id='actor_iran',
        action_category='kinetic',
        target_entities=['actor_us'],
        instruments_used=['inst_iran_proxies'],
        intended_effect='Escalate proxy attacks on US-allied positions in Syria to signal resolve and raise US intervention costs.',
        resource_cost=2,
    ),
    ActionIntent(
        actor_id='actor_us',
        action_category='information',
        target_entities=['actor_iran'],
        instruments_used=['inst_us_information'],
        intended_effect='Amplify social media campaign highlighting regime corruption to erode internal legitimacy.',
        resource_cost=1,
    ),
]

print(f'Defined {len(test_actions)} test actions:')
for i, a in enumerate(test_actions):
    print(f'  {i+1}. [{a.action_category}] {a.actor_id} -> {a.target_entities}: {a.intended_effect[:60]}...')

## Test Domain Model Selection and Base Rate Computation

In [ ]:
# Build messages for the first action and check size
action = test_actions[0]
dms = select_relevant_domain_models(spec, action)
base_rates = compute_mechanical_base_rate(dms, action, state)

messages = build_gm_messages(
    action=action,
    state=state,
    domain_models=dms,
    base_rates=base_rates,
    actor_ids=actor_ids,
    variable_ids=variable_ids,
)

total_chars = sum(len(m['content']) for m in messages)
print(f'Message count: {len(messages)}')
print(f'Total chars: {total_chars} (~{total_chars/4:.0f} tokens)')
print(f'\n--- System prompt (first 300 chars) ---')
print(messages[0]['content'][:300])
print('...')
print(f'\n--- User prompt (last 300 chars) ---')
print(messages[1]['content'][-300:])

## Test Prompt Construction and Token Count

In [ ]:
from llm_client import call_llm_structured
import uuid

GM_MODEL = 'gemini/gemini-2.5-flash'
TRACE_ID = f'nb2_gm_test_{uuid.uuid4().hex[:8]}'

results = []

for i, action in enumerate(test_actions):
    dms = select_relevant_domain_models(spec, action)
    base_rates = compute_mechanical_base_rate(dms, action, state)
    messages = build_gm_messages(
        action=action,
        state=state,
        domain_models=dms,
        base_rates=base_rates,
        actor_ids=actor_ids,
        variable_ids=variable_ids,
    )
    
    print(f'\n--- Action {i+1}/{len(test_actions)}: {action.action_category} ---')
    print(f'Calling {GM_MODEL}...')
    
    try:
        packet, call_result = call_llm_structured(
            model=GM_MODEL,
            messages=messages,
            response_model=AdjudicationPacket,
            task='wargame_gm_adjudication',
            trace_id=TRACE_ID,
            max_budget=1.0,
        )
        
        # Validate
        issues = validate_adjudication(packet, valid_var_ids, valid_actor_ids, base_rates)
        
        # Normalize if slightly off
        prob_sum = sum(o.probability for o in packet.possible_outcomes)
        if abs(prob_sum - 1.0) > 0.001:
            print(f'  Normalizing probabilities (sum was {prob_sum:.4f})')
            packet = normalize_probabilities(packet)
        
        results.append({
            'action_idx': i,
            'action': action,
            'base_rates': base_rates,
            'packet': packet,
            'issues': issues,
            'success': len(issues) == 0,
            'cost': getattr(call_result, 'cost', None),
        })
        
        # Print summary
        print(f'  Reasoning: {packet.reasoning[:120]}...')
        print(f'  Outcomes:')
        for o in packet.possible_outcomes:
            print(f'    {o.outcome_id}: {o.probability:.2f} — {o.narrative[:80]}...')
        if issues:
            print(f'  ⚠ Issues: {issues}')
        else:
            print(f'  ✓ Valid')
            
    except Exception as e:
        print(f'  ✗ FAILED: {e}')
        import traceback
        traceback.print_exc()
        results.append({
            'action_idx': i,
            'action': action,
            'base_rates': base_rates,
            'packet': None,
            'issues': [str(e)],
            'success': False,
        })

# Summary
successes = sum(1 for r in results if r['success'])
print(f'\n=== Results: {successes}/{len(results)} valid adjudications ===')

## Call the GM LLM

Run all 5 actions through the GM. Uses `llm_client.call_llm_structured()` with the AdjudicationPacket Pydantic model.

## Analyze Probability Calibration

Compare GM's probabilities to mechanical base rates.

In [ ]:
print(f'{"Action":<12} {"Outcome":<20} {"Base":>6} {"GM":>6} {"Delta":>6} {"OK?":>5}')
print('-' * 60)

all_within_tolerance = True
for r in results:
    if r['packet'] is None:
        continue
    action_label = r['action'].action_category[:10]
    for o in r['packet'].possible_outcomes:
        # Find matching base rate
        base = r['base_rates'].get(o.outcome_id)
        if base is not None:
            delta = o.probability - base
            ok = abs(delta) <= 0.15
            if not ok:
                all_within_tolerance = False
            print(f'{action_label:<12} {o.outcome_id:<20} {base:>6.2f} {o.probability:>6.2f} {delta:>+6.2f} {"✓" if ok else "✗":>5}')
        else:
            print(f'{action_label:<12} {o.outcome_id:<20} {"N/A":>6} {o.probability:>6.2f} {"":>6} {"—":>5}')

if all_within_tolerance:
    print('\n✓ All GM probabilities within ±0.15 of base rates')
else:
    print('\n⚠ Some GM probabilities deviate >0.15 from base rates — review reasoning')

## Check State Transition Reasonableness

In [ ]:
print('State transitions per outcome:\n')
for r in results:
    if r['packet'] is None:
        continue
    print(f'--- {r["action"].action_category} action ---')
    for o in r['packet'].possible_outcomes:
        if o.state_transitions:
            print(f'  {o.outcome_id}:')
            for t in o.state_transitions:
                # Check delta magnitude
                magnitude_ok = abs(t.delta) <= 0.30
                flag = '' if magnitude_ok else ' ⚠ LARGE DELTA'
                print(f'    {t.var_id}: {t.delta:+.3f}{flag}')
    print()

## Cost Analysis

In [ ]:
try:
    from llm_client import get_cost
    cost = get_cost(trace_id=TRACE_ID)
    print(f'Total cost for {len(test_actions)} GM calls: ${cost:.4f}')
    print(f'Cost per adjudication: ${cost/len(test_actions):.4f}')
    print(f'Projected cost for 20-turn game (2 actions/turn): ${cost/len(test_actions) * 40:.2f}')
except Exception as e:
    print(f'Cost query failed: {e}')
    print('(This is OK if llm_client observability is not configured)')

## Summary

**Pass criteria:**
- [ ] All 5 AdjudicationPackets parse successfully
- [ ] Probabilities sum to 1.0 in all cases (after normalization if needed)
- [ ] All var_ids exist in the scenario schema
- [ ] GM's probabilities within ±0.15 of base rates
- [ ] GM reasoning references domain models
- [ ] >80% rated plausible on manual review

**Uncertainties resolved:**
- U1: Can the GM produce calibrated probabilities?
- U2: How much context fits in the prompt?
- U8: Does base rate anchoring improve output?